# Preview: upcoming dataset-tools features

> These are **small, concrete** draft PRs with high certainty of landing roughly as shown: the pydantic dataset specs (#1528, already released), non-dask preprocessing backends (#1579), user metadata extraction (#1600), and adaptive/resizable steps (#1601).
>
> Every cell is **guarded** (detected by import/signature introspection, never by version number), so this notebook is safe to "Run All" in the default environment. To run the live demos, use the preview environment:
>
> ```bash
> pixi run -e preview jupyter lab
> ```
>
> The larger, more experimental `coffea.compute` execution refactor lives separately in **`coffea-06-experimental.ipynb`**.

In [ ]:
# Feature detection for the dataset-tools previews. Each is detected by import /
# signature introspection (never by version number), so this notebook is safe to
# "Run All" in the default environment -- missing features just print a note.
import importlib.util
import inspect
from pathlib import Path

import coffea
from coffea.dataset_tools import preprocess as _preprocess


def _has_param(func, name):
    try:
        return name in inspect.signature(func).parameters
    except (TypeError, ValueError):
        return False


HAS_PP_BACKENDS = _has_param(_preprocess, "backend")             # draft PR #1579
HAS_PP_METADATA = _has_param(_preprocess, "metadata_extractor")  # draft PR #1600
HAS_MUTABLE_STEPS = importlib.util.find_spec("coffea.dataset_tools.mutable_steps") is not None  # draft PR #1601


def preview_note(feature, pr):
    print(
        f"[preview] '{feature}' is not in this coffea build ({coffea.__version__}).\n"
        f"          It ships in draft PR {pr}. Launch the preview environment to try it:\n"
        f"            pixi run -e preview jupyter lab"
    )


# A small, network-free sample so the demos run wherever this repo is checked out.
_preview_file = Path("../columnar/data/SMHiggsToZZTo4L.root")

print(f"coffea {coffea.__version__}")
for _flag in ["HAS_PP_BACKENDS", "HAS_PP_METADATA", "HAS_MUTABLE_STEPS"]:
    print(f"  {_flag} = {globals()[_flag]}")

### 1. Pydantic dataset specifications

*Released in coffea 2026.7 (PR #1528) — the foundation the previews build on.*

A whole analysis input can now be written as a validated `pydantic` model. The top-level **`DataGroupSpec`** is a mapping of *dataset name → `DatasetSpec`*; each `DatasetSpec` holds a `files` mapping (`path → tree`) plus free-form `metadata`. You build it exactly as a user would — hand your nested dict straight to the class and pydantic validates and coerces every level (a malformed fileset fails fast with a clear error).

Below we build one small **two-dataset** group — `dy` with two files and `ttbar` with one — and reuse it for every demo in this notebook.

In [ ]:
# 1. Pydantic dataset specifications (released in coffea 2026.7, PR #1528)
# Build the spec the way a user would: hand a nested dict straight to the class.
# DataGroupSpec is a RootModel[dict[str, DatasetSpec]] -- pass the mapping in and
# pydantic validates/coerces every level (DatasetSpec, per-file specs) for you.
from coffea.dataset_tools import DataGroupSpec, DatasetSpec

# Two datasets: "dy" (2 files) and "ttbar" (1 file). We point every entry at the
# one bundled sample (using two path spellings for dy) so the later cells run
# offline; in a real analysis these would simply be distinct file paths.
_f = str(_preview_file)
_f_abs = str(_preview_file.resolve())

datagroup = DataGroupSpec(
    {
        "dy": {
            "files": {_f: "Events", _f_abs: "Events"},
            "metadata": {"process": "DY", "xsec": 6025.0},
        },
        "ttbar": {
            "files": {_f: "Events"},
            "metadata": {"process": "ttbar", "xsec": 729.0},
        },
    }
)

print("type:", type(datagroup).__name__)
for name, ds in datagroup.root.items():
    print(f"  {name}: {len(ds.files)} file(s), metadata={dict(ds.metadata)}")
    print(f"       file specs -> {[type(fs).__name__ for fs in ds.files.values()]}")

# A single DatasetSpec is just as easy to build directly:
one = DatasetSpec(files={_f: "Events"}, metadata={"process": "ttbar"})
print("standalone DatasetSpec ok:", isinstance(one, DatasetSpec))

### 2. Non-dask preprocessing backends — draft PR #1579

Released `preprocess()` builds a **dask-awkward** graph to discover file chunks. PR #1579 adds a `backend=` switch (`"iterative"`, `"futures"`, `"dask"`) so preprocessing can run with **no dask dependency**, plus a dedicated `preprocess_rntuple()` for RNTuple inputs. The backend classes (`IterativeBackend`, `FuturesBackend`, ...) are explicitly designed to plug into the `coffea.compute` refactor below.

In [ ]:
# 2. Non-dask preprocessing backends  (draft PR #1579)
from coffea.dataset_tools import preprocess

if HAS_PP_BACKENDS and _preview_file.exists():
    # Hand the DataGroupSpec straight in; the "iterative" backend needs no dask.
    available, report = preprocess(
        datagroup,
        step_size=100_000,
        save_form=False,
        backend="iterative",  # or "futures"; "dask" reproduces the legacy path
        skip_bad_files=True,
    )
    print("preprocessed with the dask-free 'iterative' backend; returns a", type(available).__name__)
    for name, ds in available.root.items():
        nsteps = sum(len(fs.steps or []) for fs in ds.files.values())
        print(f"  {name}: {len(ds.files)} file(s), {nsteps} step(s) discovered")
elif not HAS_PP_BACKENDS:
    preview_note("preprocess(backend=...)", "#1579")
else:
    print("[preview] sample file not found; skipping the live run.")

### 3. User-supplied metadata extraction — draft PR #1600

Computing per-dataset quantities such as the sum of generator weights normally means an extra pass over the files. PR #1600 adds `metadata_extractor` (called once per file on the open handle) and `metadata_reducer` (called once per dataset) hooks to `preprocess()`, folding that work into the preprocessing pass.

In [ ]:
# 3. User-supplied metadata extraction during preprocessing  (draft PR #1600)
from coffea.dataset_tools import preprocess

if HAS_PP_METADATA and _preview_file.exists():
    def per_file(file_handle):
        # runs once per file, on the open uproot file handle
        return {"nentries": int(file_handle["Events"].num_entries)}

    def per_dataset(per_file_meta):
        # reduce the per-file dicts into one dataset-level dict
        return {"nentries_total": sum(m["nentries"] for m in per_file_meta.values())}

    available, _ = preprocess(
        datagroup,
        step_size=100_000,
        save_form=False,
        backend="iterative",
        metadata_extractor=per_file,
        metadata_reducer=per_dataset,
        skip_bad_files=True,
    )
    # note dy (2 files) sums to twice ttbar (1 file)
    for name, ds in available.root.items():
        print(f"  {name} metadata after extraction: {dict(ds.metadata)}")
elif not HAS_PP_METADATA:
    preview_note("preprocess(metadata_extractor=..., metadata_reducer=...)", "#1600")
else:
    print("[preview] sample file not found; skipping the live run.")

### 4. Adaptive / resizable steps — draft PR #1601

Fixed step sizes over- or under-shoot when chunk cost varies. This **prototype** adds a resizable step generator whose size can be renegotiated mid-stream through the generator `.send()` channel (the same channel `coffea.compute`'s `Computable.gen_steps` uses), plus a `run_adaptive_steps` driver governed by a `WallTimeStepPolicy`. The API is explicitly marked unstable.

In [ ]:
# 4. Adaptive / resizable steps  (draft PR #1601, prototype -- API may change)
if HAS_MUTABLE_STEPS:
    from coffea.dataset_tools.mutable_steps import (
        resizable_steps,
        iter_dataset_steps,
        run_adaptive_steps,
        WallTimeStepPolicy,
    )

    # (a) the low-level resizable generator: renegotiate the step size mid-stream
    gen = resizable_steps(0, 1_000, 200)
    produced = [next(gen)]
    try:
        while True:
            produced.append(gen.send(100))  # after the first chunk, shrink to 100
    except StopIteration:
        pass
    print("resizable_steps, shrunk mid-stream via .send(100):")
    print(" ", produced)

    # (b) drive it from the real "dy" DatasetSpec we preprocessed above
    if HAS_PP_BACKENDS and _preview_file.exists():
        from coffea.dataset_tools import preprocess

        available, _ = preprocess(
            datagroup, step_size=100_000, save_form=False, backend="iterative", skip_bad_files=True
        )
        dy = available.root["dy"]
        steps = list(iter_dataset_steps(dy, step_size=100_000))
        print(f"\niter_dataset_steps(dy): {len(steps)} step(s); first = {steps[0]}")

        policy = WallTimeStepPolicy(target_seconds=30)
        try:
            total = run_adaptive_steps(
                dy, (lambda path, start, stop: stop - start), step_size=100_000, policy=policy
            )
            print("run_adaptive_steps(dy) -> rows processed:", total)
        except Exception as exc:  # prototype API
            print(f"run_adaptive_steps is a prototype ({type(exc).__name__}); shown for API shape.")
else:
    preview_note("coffea.dataset_tools.mutable_steps", "#1601")